# Matched LitSearch evaluation and analysis

Read README.md steps 6–9 first. Use the `tie_train` kernel for this client. Start vLLM separately in `tie_serve`; do not run training and serving on the same GPU simultaneously.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BUNDLE = Path.cwd().resolve()  # Launch Jupyter from the tie_sensitivity directory.
assert (BUNDLE / "train.py").is_file(), "Set BUNDLE to the directory containing train.py"
REPO = BUNDLE.parent.parent
ARTIFACTS = Path("/DATA/j_andrei/tie_sensitivity_artifacts")  # EDIT
def run(script, *args):
    command = [sys.executable, str(BUNDLE / script), *map(str, args)]
    print("Running:", command, flush=True)
    subprocess.run(command, check=True)


In [ ]:
RUN_NAME = "deterministic_seed42"  # Change together with the checkpoint in the serving terminal
CORPUS_REVISION = os.environ.get("LITSEARCH_REVISION", "")  # Saved 40-character corpus SHA
VLLM_VERSION = os.environ.get("TIE_VLLM_VERSION", "")  # Actual version in the serving environment
LIMIT = 5  # First smoke-test 5 queries/retriever; set 0 for the full evaluation


## Freeze inputs ONCE

Skip this cell if `evaluation_inputs/manifest.json` already exists. Do not prepare inputs separately for each condition. Set the pinned corpus revision before first use.


In [ ]:
assert len(CORPUS_REVISION) == 40, "Set CORPUS_REVISION to the saved dataset commit SHA"
run("prepare_evaluation.py", "--retrieval-root", REPO / "LitSearch/results/retrieval",
    "--corpus-revision", CORPUS_REVISION, "--output", ARTIFACTS / "evaluation_inputs")


## Start the matching server in another terminal

Use the README command (`vllm serve .../runs/$RUN_NAME/final`, matching alias, BF16, context 65536, `--generation-config vllm`). Then run the next cell. Stop/restart the server for each checkpoint and update RUN_NAME in both places. The alias check cannot detect a wrong checkpoint behind a correctly named alias.


In [ ]:
assert VLLM_VERSION, "Record the actual serving version in VLLM_VERSION"
os.environ["TIE_VLLM_VERSION"] = VLLM_VERSION
run("evaluate.py", "--run", ARTIFACTS / "runs" / RUN_NAME,
    "--inputs", ARTIFACTS / "evaluation_inputs", "--served-model", RUN_NAME,
    "--limit", LIMIT)


## Analyze only completed, full evaluations

For the pilot use `[42]` and a pilot output directory. Once all six runs are evaluated, use `[42, 2026, 3407]` and a new output directory. Fixed-random is optional and requires its matching full runs. Statistical intervals condition on the checkpoints; seed SD is separate. Nonsignificance does not establish equivalence.


In [ ]:
SEEDS = [42]
COMPARISONS = ["epoch_random"]
ANALYSIS = ARTIFACTS / "analysis_pilot"  # Must be empty/nonexistent; use analysis_full for all three seeds
run("analyze.py", "--runs", ARTIFACTS / "runs",
    "--seeds", *SEEDS, "--comparisons", *COMPARISONS, "--output", ANALYSIS)
